# 05 — MODIS on-demand download + append features to chips

Download 8-day L3m subsets for a month and append them to chip_indices.csv using your project script.


In [8]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first) from REPO_ROOT."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check, cwd=REPO_ROOT)

def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = Path(c)
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

import os
FILELISTS_8D_DIR = REPO_ROOT / "data/filelists/8d"
TMP_MODIS_ROOT = pick_first_existing("data/l3/tmp_infer", "data/l3/tmp_oci", "data/l3/tmp")
TMP_MODIS_ROOT.mkdir(parents=True, exist_ok=True)

OUT_ROOT = REPO_ROOT / "deployment/outputs/by_plant/osm_way_386838289"
MONTH_TAG = "2025-03"
CHIPS_DIR = OUT_ROOT / MONTH_TAG / "chips"
CHIP_CSV  = CHIPS_DIR / "chip_indices.csv"
require_exists(CHIP_CSV, "chip_indices.csv")

print("FILELISTS_8D_DIR:", FILELISTS_8D_DIR)
print("TMP_MODIS_ROOT:", TMP_MODIS_ROOT)
print("CHIP_CSV:", CHIP_CSV)


REPO_ROOT: /Users/ameerfiras/REDNET-ML
FILELISTS_8D_DIR: /Users/ameerfiras/REDNET-ML/data/filelists/8d
TMP_MODIS_ROOT: data/l3/tmp_infer
CHIP_CSV: /Users/ameerfiras/REDNET-ML/deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/chip_indices.csv


## 5.1 Download MODIS subset for the month


In [9]:

import re
import os
from pathlib import Path
from datetime import date, datetime, timedelta

OBPG_APPKEY = os.environ.get("OBPG_APPKEY")
if not OBPG_APPKEY:
    raise RuntimeError("Set OBPG_APPKEY before running downloads.")

FILELIST_MAP = {
    "chlor_a": Path(FILELISTS_8D_DIR) / "filelist_8d_chlor_a_filtered.txt",
    "Kd_490":  Path(FILELISTS_8D_DIR) / "filelist_8d_Kd_490_filtered.txt",
    "nflh":    Path(FILELISTS_8D_DIR) / "filelist_8d_nflh_filtered.txt",
    "sst":     Path(FILELISTS_8D_DIR) / "filelist_8d_sst.txt",
}

DATE_RANGE = re.compile(r"(\d{8})[_\-](\d{8})")
DATE_ONE   = re.compile(r"(\d{8})")

def mid_date(line: str):
    m = DATE_RANGE.search(line)
    if m:
        d0 = datetime.strptime(m.group(1), "%Y%m%d").date()
        d1 = datetime.strptime(m.group(2), "%Y%m%d").date()
        return d0 + (d1 - d0)//2
    m = DATE_ONE.search(line)
    if m:
        return datetime.strptime(m.group(1), "%Y%m%d").date()
    return None

def subset_filelist(filelist: Path, start: date, end: date, max_days=30):
    lo = start - timedelta(days=max_days)
    hi = end + timedelta(days=max_days)
    keep=[]
    for line in filelist.read_text().splitlines():
        s=line.strip()
        if not s or s.startswith("#"):
            continue
        md = mid_date(s)
        if md and lo <= md <= hi:
            keep.append(s)
    return keep

START="2025-03-01"; END="2025-03-31"; MAX_DAYS=30
start=date.fromisoformat(START); end=date.fromisoformat(END)

for prod, fl in FILELIST_MAP.items():
    require_exists(fl, f"filelist for {prod}")
    out_dir = Path(TMP_MODIS_ROOT) / str(start.year) / prod
    out_dir.mkdir(parents=True, exist_ok=True)
    lines = subset_filelist(fl, start, end, max_days=MAX_DAYS)
    subset = out_dir / f"filelist_subset_{START}_{END}.txt"
    subset.write_text("\n".join(lines) + "\n")
    sh(f'python scripts/download/obdaac_download.py --filelist "{subset}" --odir "{out_dir}" --appkey "{OBPG_APPKEY}"')



▶ python scripts/download/obdaac_download.py --filelist "data/l3/tmp_infer/2025/chlor_a/filelist_subset_2025-03-01_2025-03-31.txt" --odir "data/l3/tmp_infer/2025/chlor_a" --appkey "17ce4e76c6bb3af9d8e2f75fe57798aabe18473f"

▶ python scripts/download/obdaac_download.py --filelist "data/l3/tmp_infer/2025/Kd_490/filelist_subset_2025-03-01_2025-03-31.txt" --odir "data/l3/tmp_infer/2025/Kd_490" --appkey "17ce4e76c6bb3af9d8e2f75fe57798aabe18473f"

▶ python scripts/download/obdaac_download.py --filelist "data/l3/tmp_infer/2025/nflh/filelist_subset_2025-03-01_2025-03-31.txt" --odir "data/l3/tmp_infer/2025/nflh" --appkey "17ce4e76c6bb3af9d8e2f75fe57798aabe18473f"

▶ python scripts/download/obdaac_download.py --filelist "data/l3/tmp_infer/2025/sst/filelist_subset_2025-03-01_2025-03-31.txt" --odir "data/l3/tmp_infer/2025/sst" --appkey "17ce4e76c6bb3af9d8e2f75fe57798aabe18473f"


## 5.2 Append MODIS features into chip_indices.csv


In [12]:

MAX_DAYS = 30
CHIP_CSV_REL = Path(CHIP_CSV).resolve().relative_to(REPO_ROOT)
for prod in ["chlor_a","Kd_490","nflh","sst"]:
    modis_root = Path(TMP_MODIS_ROOT) / "2025" / prod
    sh(
      f'python scripts/HAB/preparation/append_modis_features_8d.py '
      f'--chips_csv_glob "{CHIP_CSV_REL}" --modis_root "{modis_root}" '
      f'--max_days {MAX_DAYS} --products {prod}'
    )

import pandas as pd
df = pd.read_csv(CHIP_CSV)
display(df.head(3))



▶ python scripts/HAB/preparation/append_modis_features_8d.py --chips_csv_glob "deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/chip_indices.csv" --modis_root "data/l3/tmp_infer/2025/chlor_a" --max_days 30 --products chlor_a

▶ python scripts/HAB/preparation/append_modis_features_8d.py --chips_csv_glob "deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/chip_indices.csv" --modis_root "data/l3/tmp_infer/2025/Kd_490" --max_days 30 --products Kd_490

▶ python scripts/HAB/preparation/append_modis_features_8d.py --chips_csv_glob "deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/chip_indices.csv" --modis_root "data/l3/tmp_infer/2025/nflh" --max_days 30 --products nflh

▶ python scripts/HAB/preparation/append_modis_features_8d.py --chips_csv_glob "deployment/outputs/by_plant/osm_way_386838289/2025-03/chips/chip_indices.csv" --modis_root "data/l3/tmp_infer/2025/sst" --max_days 30 --products sst


,tile,scene_id,datetime,ndwi_mean,ndwi_std,fai_mean,fai_std,rednir_mean,rednir_std,valid_px
